# 대각화와 동적 시스템

> 선형대수 14강 · 고윳값과 고유벡터

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [대각화와 동적 시스템](https://mioon1402.github.io/timeseriesdata/linalg/L14-diagonalization.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 거듭제곱이 어려운가

## 1. A = SΛS⁻¹ 유도

## 2. Aᵏ 가 붕괴한다

## 3. 대각화가 안 되는 경우

## 4. 차분방정식 — 반복하면 어디로 가나

## 5. 미분방정식과 안정성

## 6. numpy 로 확인하기

**14-1. 대각화 해보기**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[2., 1.],
              [1., 2.]])

λ, S = np.linalg.eig(A)
Λ = np.diag(λ)

print("S (고유벡터를 열로) =")
print(S)
print("\nΛ =")
print(Λ)
print("\nS Λ S⁻¹ =")
print(S @ Λ @ np.linalg.inv(S))
print("\nA 와 같은가:", np.allclose(S @ Λ @ np.linalg.inv(S), A))

**14-2. Aᵏ 가 정말 붕괴하는가**

In [ ]:
k = 10
직접 = np.linalg.matrix_power(A, k)
대각화 = S @ np.diag(λ ** k) @ np.linalg.inv(S)

print(f"A^{k} 직접 계산 =")
print(직접)
print(f"\nS Λ^{k} S⁻¹ =")
print(대각화)
print("\n같은가:", np.allclose(직접, 대각화))
print()
print("λ^k =", λ ** k, "  ← 대각 원소를 k제곱한 것뿐")

**14-3. 대각화가 안 되는 행렬**

In [ ]:
전단 = np.array([[1., 1.],
                 [0., 1.]])

w, v = np.linalg.eig(전단)
print("고윳값 =", w, "  ← 1 이 두 번 (중복)")
print("고유벡터 =")
print(v)
print("\n고유벡터 행렬의 랭크 =", np.linalg.matrix_rank(v), "/ 2")
print("→ 독립인 고유벡터가 1개뿐이라 S⁻¹ 가 없다. 대각화 불가.")
print()
print("그래도 거듭제곱은 규칙적이다:")
for k in [1, 2, 5]:
    print(f"  전단^{k} =", np.linalg.matrix_power(전단, k).ravel())
print("→ 위쪽 값이 k 로 '선형' 증가. 지수가 아니라서 λ^k 로 설명이 안 된다.")

**14-4. 마르코프 체인 — 장기 예측**

In [ ]:
# 열의 합이 1 인 행렬. 각 열이 '그 상태에서 어디로 갈 확률'
M = np.array([[0.9, 0.2],      # 도시 → 도시,  시골 → 도시
              [0.1, 0.8]])     # 도시 → 시골,  시골 → 시골

print("열의 합 =", M.sum(axis=0), "  ← 확률이므로 각 열이 1")
w, v = np.linalg.eig(M)
print("고윳값 =", w, "  ← 하나가 정확히 1")
print()

u = np.array([1., 0.])          # 전부 도시에서 시작
for t in [0, 1, 5, 20, 100]:
    상태 = np.linalg.matrix_power(M, t) @ u
    print(f"  t={t:3d}:  도시 {상태[0]:.4f}  시골 {상태[1]:.4f}")

**14-5. 정상상태 = 고윳값 1의 고유벡터**

In [ ]:
i = np.argmin(np.abs(w - 1))
정상 = np.real(v[:, i])
정상 = 정상 / 정상.sum()          # 확률이므로 합을 1로

print("정상상태 =", 정상)
print("M @ 정상 =", M @ 정상, "  ← 변하지 않는다. 그래서 '정상' 상태")
print()
print("시작점을 바꿔도 같은 곳으로 가는가:")
for 시작 in [np.array([1., 0.]), np.array([0., 1.]), np.array([0.3, 0.7])]:
    끝 = np.linalg.matrix_power(M, 200) @ 시작
    print(f"  {시작} → {np.round(끝, 6)}")
print()
print("→ 두 번째 고윳값 0.7 의 성분이 0.7²⁰⁰ ≈ 0 으로 사라졌기 때문")

**14-6. 미분방정식과 안정성**

In [ ]:
from scipy.linalg import expm

시스템들 = {
    "안정   (Re λ < 0)": np.array([[-1., 1.], [0., -2.]]),
    "불안정 (Re λ > 0)": np.array([[1., 0.], [0., 0.5]]),
    "진동   (Re λ = 0)": np.array([[0., -2.], [2., 0.]]),
}

u0 = np.array([1., 1.])
for 이름, B in 시스템들.items():
    λs = np.linalg.eigvals(B)
    crit = λs.real.max()
    print(f"{이름}  λ = {np.round(λs, 3)}   최대 실수부 = {crit:+.2f}")
    for t in [0., 1., 3.]:
        u = expm(B * t) @ u0
        print(f"     t={t:.0f}  u = {np.round(u, 4)}   ‖u‖ = {np.linalg.norm(u):.4f}")
    print()

**14-7. expm 도 대각화로 계산된다**

In [ ]:
from scipy.linalg import expm

직접 = expm(A)
대각화 = S @ np.diag(np.exp(λ)) @ np.linalg.inv(S)

print("expm(A) =")
print(직접)
print("\nS e^Λ S⁻¹ =")
print(대각화)
print("\n같은가:", np.allclose(직접, 대각화))
print()
print("무한급수를 직접 더해도 같은가 (20항까지):")
from math import factorial
급수 = sum(np.linalg.matrix_power(A, n) / factorial(n) for n in range(20))
print(np.round(급수, 6))
print("일치:", np.allclose(급수, 직접))

**14-8. 연습문제**

In [ ]:
# 문제 1. [[3,0],[0,0.5]] 를 100번 반복하면 (1,1) 은 어디로 갈까요?

# 문제 2. 마르코프 행렬 [[0.7,0.4],[0.3,0.6]] 의 정상상태를 구하세요.

# 문제 3. du/dt = Au 에서 A = [[-2,0],[0,3]] 은 안정할까요?
#         (힌트: 한 방향만 발산해도 전체는 불안정)

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — x축 성분만 3^100 으로 폭발, y축은 0.5^100 으로 소멸
A1 = np.array([[3., 0.], [0., 0.5]])
u = np.linalg.matrix_power(A1, 100) @ np.array([1., 1.])
print("문제 1:", u)
print("        방향 =", u / np.linalg.norm(u), " ← 사실상 (1,0). 지배 고유벡터")

# 문제 2
M2 = np.array([[0.7, 0.4], [0.3, 0.6]])
w2, v2 = np.linalg.eig(M2)
i2 = np.argmin(np.abs(w2 - 1))
st = np.real(v2[:, i2]); st = st / st.sum()
print("\n문제 2: 정상상태 =", st, "  확인:", M2 @ st)

# 문제 3 — 고윳값 -2 와 +3. 하나라도 양수면 불안정
A3 = np.array([[-2., 0.], [0., 3.]])
print("\n문제 3: λ =", np.linalg.eigvals(A3))
print("        최대 실수부 =", np.linalg.eigvals(A3).real.max(), "> 0 → 불안정")
print("        x 방향은 잦아들지만 y 방향이 폭발한다")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)